# Demo: Bag-of-Words Model
In this demo, we will explore the Bag-of-Words (BoW) model. The Bag-of-Words model converts an input token sequence into a high-dimensional count (or frequency) vector by first building an explicit vocabulary $\mathcal{V}$. It is simple, interpretable, and effective for many classic NLP tasks.

> __Model:__ Suppose you have a corpus of text documents, e.g., a collection of news articles, product reviews, or any other text data, and you want to represent each document as a vector of word counts, or perhaps word frequencies. The Bag-of-Words model does this by counting how many times each word appears in the document, _ignoring the order of words_.

First, we build the base vocabulary $\mathcal{V}$ (words, sub-words, characters, etc) from the corpus, $\mathcal V = \{w_1, w_2, \dots, w_V\}$ where the number of elements in the vocabulary is $V = |\mathcal V_{\text{base}}|$. Optionally we add some special tokens, e.g., `<unk>` for out-of-vocabulary words, `<bos>`, `<eos>`, etc., making the full vocabulary size $N_{\mathcal V}=V+K$, where $K$ is the number of special tokens. 

Let's take a look at a simple algorithm to construct a Bag-of-Words model from a sequence of tokens $\mathbf X=(x_1,\dots,x_n)\in\mathcal V^n$. The algorithm will produce a count vector $\mathbf C\in\mathbb R^{N_{\mathcal V}}$ that contains the counts of each token in the sequence $\mathbf X$ (raw counts version).

__Initialization:__ Given an array of tokens $\mathbf{X} = \{x_1, x_2, \ldots, x_n\}$, and a vocabulary size $N_{\mathcal V}$, initialize a result vector $\mathbf C = \mathbf 0\in\mathbb R^{N_{\mathcal V}}.$

For each token $x\in\mathbf X$ __do__:
1. Check if $x$ is in the vocabulary $\mathcal V$. 
    - Set $j\gets \texttt{ID}(x)$, if $x\in\mathcal V$, otherwise $j\gets \texttt{ID}(\text{<unk>})$ token.
2. Increment the count:  $C_j \;\mathrel{+}= 1.$

> **Variant (term frequency–inverse document frequency, tf-idf):**  After collecting raw token counts $C_j$, you can weight each entry by:  
> $$\mathrm{tfidf}_j = \frac{C_j}{|\mathbf{X}|} \times \log\left(\frac{N_{\text{docs}}}{1 + \text{df}_j}\right),$$  
> where $\text{df}_j$ is the number of messages containing token $x_j$, $N_{\text{docs}}$ is the total number of messages in the dataset, and $|\mathbf{X}|$ is the number of tokens in the current message (excluding padding). This gives more weight to rare words across messages, reducing the impact of common words.

**Example**  
```text
Tokens: ["Hello", "world!", "This", "is", "a", "test", "."]
Vocabulary: ["<unk>", ".", "Hello", "a", "is", "test", "This", "world!"]
V = 8

Raw counts vector C (by index):
 [0, 1, 1, 1, 1, 1, 1, 1]
````

Here, `<unk>` = 0 (no OOV), `"."`=1, `"Hello"`=1, …, `"world!"`=1.

### Learning objectives
This demo will familiarize students with the Bag-of-Words approach to text representation, which differs fundamentally from both the explicit tokenization and feature hashing methods we've explored. In this notebook, we'll see how bag-of-words focuses on word presence and frequency rather than position or hash collisions. The main tasks are:
* __Setup and data exploration:__ Load and examine the SMS spam/ham dataset to understand our text corpus
* __Build raw count vectors:__ Implement the basic bag-of-words algorithm that counts word occurrences
* __Compute tf-idf weighting:__ Enhance our representation by weighting words based on their rarity across the entire dataset

By the end, you'll understand when bag-of-words is the right choice for text analysis and how it trades positional information for interpretability and computational simplicity.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.
* The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [1]:
include("Include.jl")

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

### Data
Let's load a public dataset of SMS messages that have been curated as either __spam__ or __ham__. The dataset we'll use is [publically available on Kaggle](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset?resource=download) and is also discussed in the publications:
* Almeida, T. & Hidalgo, J. (2011). SMS Spam Collection [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84.
* Tiago A. Almeida, José María G. Hidalgo, and Akebo Yamakami. 2011. Contributions to the study of SMS spam filtering: new collection and results. In Proceedings of the 11th ACM symposium on Document engineering (DocEng '11). Association for Computing Machinery, New York, NY, USA, 259–262. https://doi.org/10.1145/2034691.2034742


We've packaged the spam/ham dataset in [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). We'll load the dataset using [the `MyMSSpamHamCorpus(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MyMSSpamHamCorpus) which returns [a `MyMSSpamHamRecordCorpusModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySMSSpamHamRecordCorpusModel) with the fields:
* The `records::Dict{Int, MyMSSpamHamRecordModel}` field holds the original records data as a dictionary, where the keys of the dictionary correspond to the headline index, and the values are [instances of the `MyMSSpamHamRecordModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySMSSpamHamRecordModel). Each record has the following fields:
    * `isspam`: has a value of `1` if the record is spam; otherwise, `0.`
    * `message`: the message of the SMS, unstructured text

* The `tokens::Dict{String, Int64}` field holds the vocabulary computed over the __entire dataset__ as a dictionary, where the dictionary's keys are the tokens (words) and the values of the index of the word. We assemble the `tokens` dictionary in alphabetical order. 
* The `inverse::Dict{Int64, String}` field is the inverse of the `tokens` dictionary, where the keys are the token indexes and the values are the tokens (words).

Let's call [the `MySMSSpamHamCorpus(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MySMSSpamHamCorpus) to load the spam/ham dataset and assign it to the `corpusmodel:MySMSSpamHamRecordCorpusModel` variable.

In [2]:
corpusmodel = MySMSSpamHamCorpus();

Let's check out the fields of the corpus model, starting with the `tokens::Dict{String,Int64}` field. This is a dictionary with the keys being a word from the vocabulary of the SMS message, pointing to an ID value (sorted in alphabetical order). This collection is 0-based.

In [3]:
corpusmodel.tokens

Dict{String, Int64} with 9998 entries:
  "Goal"        => 1787
  "youd"        => 9959
  "Gd"          => 1765
  "enjoy"       => 5605
  "Badrith"     => 1036
  "Bani"        => 1041
  "TAKE"        => 3396
  "chocolate"   => 4896
  "everywhere"  => 5667
  "fight"       => 5796
  "08714712394" => 111
  "princess"    => 7973
  "helping"     => 6275
  "whose"       => 9739
  "Many"        => 2445
  "favor"       => 5762
  "hurried"     => 6416
  "loads"       => 6919
  "8000930705"  => 658
  ⋮             => ⋮

Next, lets check out the `inverse::Dict{Int,String}` field. This is a dictionary where the keys are the token IDs while the values are the words (tokens) computed by parsing all the SMS messages (both spam and ham). This collection is 0-based.

In [4]:
corpusmodel.inverse[5]

"0089my"

Finally, let's look at the records. These hold the original text in the `message::String` field, and the label, i.e., whether the message is spam or ham in the `isspam::Bool` field. The records are stored in the `records::Dict{Int, MyMSSpamHamRecordModel}` field, where the keys of the dictionary correspond to the message index, and the values are [instances of the `MyMSSpamHamRecordModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySMSSpamHamRecordModel).

In [5]:
corpusmodel.records

Dict{Int64, MySMSSpamHamRecordModel} with 5573 entries:
  4986 => MySMSSpamHamRecordModel(false, "Dont let studying stress you out L8r")
  4700 => MySMSSpamHamRecordModel(false, "Yes fine")
  4576 => MySMSSpamHamRecordModel(false, "Not directly behind Abt 4 rows behind…
  2288 => MySMSSpamHamRecordModel(false, "I have had two more letters from I wi…
  1703 => MySMSSpamHamRecordModel(false, "Can come my room but cannot come my h…
  1956 => MySMSSpamHamRecordModel(false, "Good night Am going to sleep")
  2350 => MySMSSpamHamRecordModel(false, "That seems unnecessarily affectionate…
  3406 => MySMSSpamHamRecordModel(false, "Jus chillaxin")
  2841 => MySMSSpamHamRecordModel(false, "BABE I miiiiiiissssssssss you I need …
  2876 => MySMSSpamHamRecordModel(false, "Idk You keep saying that youre not")
  687  => MySMSSpamHamRecordModel(false, "Have you finished work yet")
  185  => MySMSSpamHamRecordModel(false, "He will")
  1090 => MySMSSpamHamRecordModel(true, "You are awarded a SiPix Digital

## Tokenize the SMS Messages
In this task, we'll use the `tokens::Dict{String, Int64}` dictionary, to tokenize messages in our dataset, i.e., convert a text representation into a numerical vector representation.

> __Idea__: To better understand how this works, let's first examine a single (random) record and tokenize it.  We'll select a random record from the `number_of_records::Int64` possible records [using the built-in `rand(...)` method](https://docs.julialang.org/en/v1/stdlib/Random/#Base.rand), and store it in the `random_test_record::MyMSSpamHamRecordModel` variable

Select a random record:

In [6]:
random_test_record = let 
    number_of_records = length(corpusmodel.records);    # how many records do we have?
    random_index = rand(1:number_of_records);           # pick a random index
    corpusmodel.records[random_index]                   # get the record at that index
end

MySMSSpamHamRecordModel(false, "Hi the way I was with u 2day")

Call [the `tokenize(...)` method exported by the `VLDataScienceMachineLearningPackage.jl` package](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/text/#VLDataScienceMachineLearningPackage.tokenize), which takes the `message::String` that we want to tokenize, and our vocabulary stored in the `tokens::Dict{String, Int64}` dictionary and returns a token vector.

What happens when we tokenize the message of the `random_test_record::MySMSSpamRecordModel` record?

In [7]:
tv = tokenize(random_test_record.message, corpusmodel.tokens)

10-element Vector{Int64}:
 9993
 1941
 9158
 9659
 2004
 9640
 9783
 9418
  418
 9994

Ok, so we get back a vector of integers, where each integer corresponds to a token in the `tokens::Dict{String, Int64}` dictionary. Let's show this in a table format, where the first column is the token index and the second column is the token itself.

`Unhide` the code cell below to see how we generated a token table using [the `pretty_table(...)` method exported by the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl).

In [8]:
let

    # initialize -
    df = DataFrame(); # holds the text data

    for i ∈ eachindex(tv)
        
        # get the token
        token = corpusmodel.inverse[tv[i]]; # use the inverse dictionary to get the token string from the token index

        # add to the dataframe
        row_data = (
            token_index = i,
            token = tv[i],
            word = token
        )
        push!(df, row_data);
    end

    # show the dataframe using PrettyTables -
    pretty_table(df, header = ["Index", "Token ID", "Word"], tf = tf_simple)
end

======== ========== ========
  Index   Token ID    Word 
======== ========== ========
      1       9993   <bos>
      2       1941      Hi
      3       9158     the
      4       9659     way
      5       2004       I
      6       9640     was
      7       9783    with
      8       9418       u
      9        418    2day
     10       9994   <eos>
======== ========== ========


### Compute the maximum pad length
Not every message has the same length, but we (may) want the token vectors to have the same size. Thus, we'll find the longest vectors in the dataset and pad the token vectors to that length. However, before we address sequence length, let's briefly consider the vocabulary we've created. Our corpus contains thousands of unique words, each mapped to a distinct integer.

This raises an important design question: __how large should our vocabulary be?__

A larger vocabulary captures more linguistic nuance, distinguishing between words like "happy," "joyful," and "ecstatic" as separate concepts. However, larger vocabularies come with costs: they require more memory for storing token mappings, create larger sets of model parameters (since each word needs its own learned representation when we construct embeddings), and can lead to overfitting on rare words that appear infrequently.

Conversely, a smaller vocabulary is more computationally efficient but might lose important distinctions by mapping different words to the same `<unk>` token. The optimal vocabulary size depends on your specific application, dataset characteristics, and computational constraints. For the ham/spam dataset, let's check our actual vocabulary size and see how it compares to the number of unique messages.

In [9]:
println("Our SMS spam dataset vocabulary contains $(length(corpusmodel.tokens)) unique tokens")
println("The dataset contains $(length(corpusmodel.records)) messages")

Our SMS spam dataset vocabulary contains 9998 unique tokens
The dataset contains 5573 messages


Ok. Back to finding the pad length. First, iterate through each message, compute its size, and then save this length if it is longer than we've seen before. We'll print out some information along the way to see how this works.

In [10]:
max_pad_length = let

    # initialize -
    number_of_records = corpusmodel.records |> length; # how many records do we have?
    max_pad_length = 0; # initialize: we have 0 length
    
    # test the length of each headline
    for i ∈ 1:number_of_records
        test_record_length = tokenize(corpusmodel.records[i].message, corpusmodel.tokens) |> length; # tokenize, and calc the number of tokens
        if (test_record_length > max_pad_length)
            max_pad_length = test_record_length; # we've found a new longest headline!
            println("Found a new longest headline: $(i) with length: $(max_pad_length)"); # show the record number and length
        end
    end
    max_pad_length
end;

Found a new longest headline: 1 with length: 6
Found a new longest headline: 2 with length: 8
Found a new longest headline: 3 with length: 30
Found a new longest headline: 6 with length: 31
Found a new longest headline: 14 with length: 39
Found a new longest headline: 32 with length: 43
Found a new longest headline: 335 with length: 45
Found a new longest headline: 712 with length: 54
Found a new longest headline: 1086 with length: 173


So, the maximum length message length is:

In [11]:
println("The maximum pad length is: $(max_pad_length) tokens.")

The maximum pad length is: 173 tokens.


Now that we know the maximum length of the token vectors, we can pad the shorter vectors with the `<pad>` token to ensure that all vectors __have the same length__. This is important for training machine learning models, as they typically require fixed-size input vectors.

> __Alternatives:__ While padding to a fixed length is the most straightforward approach, other methods exist for handling variable-length sequences. These include dynamic batching (grouping sequences of similar lengths), attention masking (teaching models to ignore padding tokens), and recurrent architectures that can naturally handle sequences of varying lengths. For this introduction, fixed-length padding provides the clearest path to understanding the fundamental concepts.

We'll use `right-padding` (adding padding tokens at the end) and will store the tokenized records for each headline in the `token_record_dictionary::Dict{Int64, Array{Int64,1}}` dictionary, where the keys of this dictionary are the record indexes, and the values are the tokenized records (which are of type `Array{Int64,1}`).

In [12]:
token_record_dictionary = let

    # initialize -
    number_of_records = corpusmodel.records |> length; # how many records do we have?
    token_record_dictionary = Dict{Int64, Array{Int64,1}}(); # dictionary to hold the tokenized records

    # tokenize each record, store result in the dictionary
    for i ∈ 1:number_of_records
        token_record_dictionary[i] = tokenize(corpusmodel.records[i].message, corpusmodel.tokens, 
                pad = max_pad_length); # tokenize, and pad to max length
    end

    token_record_dictionary;
end

Dict{Int64, Vector{Int64}} with 5573 entries:
  4986 => [9993, 1467, 6849, 8935, 8916, 9954, 7622, 2228, 9996, 9996  …  9996,…
  4700 => [9993, 3960, 5817, 9996, 9996, 9996, 9996, 9996, 9996, 9996  …  9996,…
  4576 => [9993, 2652, 5367, 4458, 832, 499, 8340, 4458, 9996, 9996  …  9996, 9…
  2288 => [9993, 2004, 6238, 6188, 9407, 7272, 6852, 5966, 2004, 9754  …  9996,…
  1703 => [9993, 1228, 4983, 7338, 8328, 4681, 4738, 4983, 7338, 6373  …  9996,…
  1956 => [9993, 1795, 7424, 886, 6087, 9265, 8663, 9996, 9996, 9996  …  9996, …
  2350 => [9993, 3510, 8475, 9460, 4067, 9996, 9996, 9996, 9996, 9996  …  9996,…
  3406 => [9993, 2159, 4889, 9996, 9996, 9996, 9996, 9996, 9996, 9996  …  9996,…
  2841 => [9993, 952, 2004, 7187, 9954, 2004, 7386, 9954, 2004, 5120  …  9996, …
  2876 => [9993, 2061, 3973, 6689, 8417, 9150, 9970, 7465, 9996, 9996  …  9996,…
  687  => [9993, 1910, 9954, 5823, 9823, 9940, 9996, 9996, 9996, 9996  …  9996,…
  185  => [9993, 1913, 9754, 9996, 9996, 9996, 9996, 9996, 9996

## Compute the Bag of Words
Now that we have the tokenized records, we can compute the Bag of Words frequency vectors. This approach represents yet another philosophy for converting text into numerical form. Unlike the explicit tokenization we explored earlier (which preserves word order) or the feature hashing approach (which uses fixed-size vectors with potential collisions), bag-of-words focuses purely on word presence and frequency.

The key insight behind bag-of-words is that for many text analysis tasks, the specific order of words matters less than which words appear and how often. This makes it particularly effective for tasks like spam detection, sentiment analysis, or document classification, where the presence of certain keywords often carries more signal than their precise arrangement.

Let's start with the raw counts version, which is simpler and easier to understand, before moving to the more sophisticated tf-idf variant that weights words by their rarity across the entire dataset.

Let's implement a simple algorithm to construct a Bag-of-Words model from a sequence of tokens $\mathbf X=(x_1,\dots,x_n)\in\mathcal V^n$. The algorithm will produce a count vector $\mathbf C\in\mathbb R^{N_{\mathcal V}}$ that contains the counts of each token in the sequence $\mathbf X$ (raw counts version).

__Initialization:__ Given an array of tokens $\mathbf{X} = \{x_1, x_2, \ldots, x_n\}$, and a vocabulary size $N_{\mathcal V}$, initialize a result vector $\mathbf C = \mathbf 0\in\mathbb R^{N_{\mathcal V}}.$

For each token $x\in\mathbf X$ __do__:
1. Check if $x$ is in the vocabulary $\mathcal V$. 
    - Set $j\gets \texttt{ID}(x)$, if $x\in\mathcal V$, otherwise $j\gets \texttt{ID}(\text{<unk>})$ token.
2. Increment the count:  $C_j \;\mathrel{+}= 1.$

Let's implement this and save the raw counts in the `raw_counts_dictionary::Dict{Int64, Array{Int64,1}}` dictionary, where the keys of this dictionary are the record indexes, and the values are the raw counts vectors (which are of type `Array{Int64,1}`).

In [15]:
raw_counts_dictionary = let

    # initialize -
    raw_counts_dictionary = Dict{Int64, Array{Int64,1}}(); # dictionary to hold the raw counts
    number_of_records = corpusmodel.records |> length; # how many records do we have?
    unk_token_id = corpusmodel.tokens["<unk>"]; # get the ID for the <unk> token

    for i ∈ 1:number_of_records

        # initialize the count vector
        count_vector = zeros(Int64, length(corpusmodel.tokens)); # create a zero vector of the same length as the vocabulary

        # get the tokenized record
        tokenized_record = token_record_dictionary[i];

        # count the tokens
        for j ∈ eachindex(tokenized_record)

            # get the token index
            token_index = tokenized_record[j];

            # check if the token is in the vocabulary
            if haskey(corpusmodel.inverse, token_index)
                # increment the count for this token (convert to 1-based indexing)
                count_vector[token_index+1] += 1; 
            else
                # if not in vocabulary, increment the <unk> token count
                count_vector[unk_token_id+1] += 1; 
            end
        end

        # store the counts in the dictionary
        raw_counts_dictionary[i] = count_vector;
    end

    # return -
    raw_counts_dictionary;
end

Dict{Int64, Vector{Int64}} with 5573 entries:
  4986 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 165, 0]
  4700 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 170, 0]
  4576 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 165, 0]
  2288 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 150, 0]
  1703 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 157, 0]
  1956 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 166, 0]
  2350 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 168, 0]
  3406 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 170, 0]
  2841 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 152, 0]
  2876 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 165, 0]
  687  => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 167, 0]
  185  => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 170, 0]
  1090 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …

__Ok, what are we seeing here?__ The raw counts dictionary holds a record index as the key, and the value is a vector of integers that represent the raw counts of each token in the vocabulary. The length of this vector is equal to the size of the vocabulary, $N_{\mathcal V} + K$, where $K$ is the number of special tokens (e.g., `<unk>`, `<pad>`, etc.).

Let's dig into this a little bit, and look at the first record. We'll do a bunch of check to make sure we are doing what we think we are doing.

In [16]:
raw_counts_dictionary[1]

9998-element Vector{Int64}:
   0
   0
   0
   0
   0
   0
   0
   0
   0
   0
   ⋮
   0
   0
   0
   0
   1
   1
   0
 168
   0

First, how long is the raw counts vector for the first record? This should be equal to the size of the vocabulary plus the number of special tokens, $N_{\mathcal V} + K$. In our case, when we built the vocabulary we included the special tokens, so we can just check the length of the tokens dictionary.

In [19]:
@assert length(corpusmodel.tokens) == length(raw_counts_dictionary[1]); # check: length correct?

Next, can we recreate the tokenized record from the raw counts vector? Again, let's check this by looking at the first record.

> __Idea:__ The non-zero elements of the raw counts vector correspond to the tokens in the tokenized record. We can iterate through the raw counts vector and find the indices of the non-zero elements, which will give us the token IDs. Then we can use the `inverse::Dict{Int64, String}` dictionary to get the actual tokens.

Let's implemment this idea, and make a table (using [the `pretty_table(...)` method exported by the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl)) that shows the token IDs and the corresponding tokens for the first record.

In [30]:
let
    
    # initialize -
    df = DataFrame(); # holds the token IDs and tokens for the table
    actual_message_tokens = token_record_dictionary[1]; # get the actual message for the first record
    C = raw_counts_dictionary[1]; # get the raw counts vector for the first record

    token_ids = findall(x -> x > 0, C); # find the indices of the non-zero elements
    tokens = [corpusmodel.inverse[token_id - 1] for token_id in token_ids]; # get the actual tokens. why -1? Token IDs are 0-based, but the indices in the raw counts vector are 1-based.

    # build a table -
    for i ∈ eachindex(token_ids)
        row_data = (
            i = i, # index of the token
            recovered_token = tokens[i],
            token_id = token_ids[i] - 1, # convert to 0-based index
            original_tokens = actual_message_tokens[i] |> j-> corpusmodel.inverse[j] # get the original token from the tokenized record
        )
        push!(df, row_data);
    end
    pretty_table(df, tf = tf_simple)
end

======== ================= ========== =================
      i   recovered_token   token_id   original_tokes 
  Int64            String      Int64           String 
======== ================= ========== =================
      1                Go       1786            <bos>
      2            jurong       6660               Go
      3             point       7865            until
      4             until       9468           jurong
      5             <bos>       9993            point
      6             <eos>       9994            <pad>
      7             <pad>       9996            <pad>
======== ================= ========== =================


This discovery reveals the fundamental characteristic of the Bag-of-Words model: we've deliberately thrown away word order information. The recovered tokens are the same as the original tokenized record, but their sequence has been lost. This represents a conscious trade-off in our text representation strategy.

This loss of order is both a limitation and a strength. It's a limitation because we can't distinguish between "the dog bit the man" and "the man bit the dog", both would produce identical bag-of-words vectors. However, it's also a strength because it makes our representation robust to variations in phrasing. Whether someone writes a spammy phrase like "cheap viagra now" or "viagra cheap now", the bag-of-words representation remains the same, which is often exactly what we want for tasks like spam detection.

This shift from sequential thinking to presence-based thinking is crucial for understanding when bag-of-words is the right tool. If word order matters for your task (like language translation or grammar checking), you'll need sequence-preserving approaches. But if you're trying to categorize content, detect sentiment, or identify topics, the presence and frequency of specific words often carries more information than their arrangement.

### Term frequency–inverse document frequency
After collecting the raw token counts $C_j$, for each message, we can weigh the frequency $f_{j}$ of each entry by $w_{j}$:  
$$\mathrm{tfidf}_j = \underbrace{\frac{C_j}{|\mathbf{X}|}}_{f_{j}} \times \underbrace{\log\left(\frac{N_{\text{docs}}}{1 + \text{df}_j}\right)}_{w_j},$$  
where $\text{df}_j$ is the number of documents containing token $x_j$, the term $N_{\text{docs}}$ is the total number of documents, and $|\mathbf{X}|$ is the number of tokens in the document we are looking at. This gives more weight to rare words across documents, reducing the impact of common words. Let's compute the tf-idf score over the entire dataset.

Let's break this computation down into steps, first let's compute the $f_{j}$ term, which is the frequency of each token in the document. This is simply the raw counts vector divided by the number of tokens in the document.

In [25]:
frequency_dictionary = let

    # initalize -
    frequency_dictionary = Dict{Int64, Array{Float64, 1}}(); # dictionary to hold the frequencies
    number_of_messages = length(corpusmodel.records); # how many messages do we have?
    L = max_pad_length; # maximum pad length

    for i ∈ 1:number_of_messages
        C = raw_counts_dictionary[i]; # get the raw counts vector for this record
        f = C ./ L; # calculate the frequency vector (raw counts divided by actual message length)

        frequency_dictionary[i] = f;
    end

    frequency_dictionary; # return the frequency dictionary
end

Dict{Int64, Vector{Float64}} with 5573 entries:
  4986 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  4700 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  4576 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2288 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  1703 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  1956 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2350 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  3406 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2841 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2876 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  687  => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  185  => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  

Next, let's compute the weight $w_{j}$ terms, which is the inverse document frequency. 

In [32]:
weight_vector, C, df = let 
    
    # initalize -
    number_of_messages = length(corpusmodel.records); # how many messages do we have?
    number_of_tokens = length(corpusmodel.tokens); # how many tokens do we have?
    N_docs = number_of_messages; # total number of documents (messages)

    # convert to count array -
    C = zeros(Int64, number_of_messages, number_of_tokens); # initialize a count array for the document frequency
    for i ∈ 1:number_of_messages
        for j ∈ 1:number_of_tokens
            C[i, j] = raw_counts_dictionary[i][j]; # fill the count array with the raw counts
        end
    end

    # compute the df -
    df = sum(C, dims = 1); # sum over the rows to get the document frequency for each token
    
    # compute the weights -
    w = zeros(Float64, number_of_tokens);
    for j ∈ 1:number_of_tokens
        w[j] = log(N_docs / (1 + df[j])); # compute the weight for this token
    end

    w, C, df
end;

In [30]:
df

1×9998 Matrix{Int64}:
 4  1  3  2  2  1  1  1  2  1  1  1  2  …  1  1  1  5573  5573  0  890883  7

What's happening in the weight vector reveals the power of tf-idf weighting. Some highly used tokens have very small or even negative weights, which might seem counterintuitive at first. For example, the `<pad>` token has the smallest weight and it's negative! This makes perfect sense when you think about it, padding tokens appear in almost every message but carry no information about whether a message is spam or ham. The tf-idf weighting automatically identifies these uninformative tokens and reduces their influence.

This automatic filtering of common, uninformative words is one of tf-idf's key strengths. Words that appear in most documents get low weights because they don't help distinguish between different types of content. Conversely, words that appear in only a few documents get high weights because they're potentially very informative about those specific documents.

In [31]:
corpusmodel.tokens["<pad>"] |> i -> weight_vector[i+1] # get the weight for the <pad> token

-5.074280719610995

In [67]:
argmin(weight_vector) |> i -> corpusmodel.inverse[i-1] # get the token with the minimum weight

"<pad>"

The largest token weight is for the `<mask>` token, which seems counterintuitive at first. However, its $df_j = 0$ as it is only used in the masked language modeling task, which is not present in this dataset. Thus, the inverse document frequency is very large, leading to a large weight. But doesn't that mean that the `<mask>` token is very important? Not really, becuasse if will have a low frequency in the document, and thus the tf-idf score will be low. 

In [68]:
argmax(weight_vector) |> i-> corpusmodel.inverse[i-1] # get the token with the maximum weight

"<mask>"

Finally, we can compute the tf-idf score for each token in the document by multiplying the frequency and weight terms together. We'll store the tf-idf scores in a new dictionary, `tfidf_dictionary::Dict{Int64, Array{Float64,1}}`, where the keys are the record indexes and the values are the tf-idf vectors (which are of type `Array{Float64,1}`).

In [27]:
tfidf_dictionary = let
    
    # intialize -
    tfidf_dictionary = Dict{Int64, Array{Float64, 1}}();
    number_of_messages = length(corpusmodel.records); # how many messages do we have?

    for i ∈ 1:number_of_messages
        f = frequency_dictionary[i]; # get the frequency vector for this record
        w = weight_vector; # get the weight vector

        # compute the tf-idf vector -
        tfidf_vector = f .* w; # element-wise multiplication of frequency and weight vectors

        tfidf_dictionary[i] = tfidf_vector; # store the tf-idf vector in the dictionary
    end

    tfidf_dictionary;
end

Dict{Int64, Vector{Float64}} with 5573 entries:
  4986 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  4700 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  4576 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2288 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  1703 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  1956 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2350 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  3406 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2841 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2876 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  687  => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  185  => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  

If we rank the tokens by their tf-idf scores, and cut off negative scores, what do we recover?


In [28]:
findall(x-> x>0, tfidf_dictionary[1]) .|> i-> corpusmodel.inverse[i-1]

4-element Vector{String}:
 "Go"
 "jurong"
 "point"
 "until"

Perfect! We recover the original tokenized record, albeit out of order. This demonstrates the fundamental nature of the Bag-of-Words model: it preserves word content while discarding word sequence.

## Summary: Understanding Bag-of-Words in Context

In this demo, we've seen how bag-of-words offers a different philosophy for text representation compared to the other methods we've studied. Where explicit tokenization preserves sequence and feature hashing trades precision for computational efficiency, bag-of-words trades sequence information for interpretability and content focus.

The tf-idf enhancement makes this approach particularly powerful by automatically identifying which words are most informative. Common words like "the," "and," or our padding tokens get low weights, while rare words that might be indicative of spam (or ham) get higher weights. This automatic feature selection is one reason why bag-of-words remains popular for many text classification tasks.

When should you reach for bag-of-words? It excels when word presence and frequency matter more than word order - tasks like document classification, spam detection, sentiment analysis, or topic modeling. It's also highly interpretable: you can easily see which words contributed most to a classification decision, making it valuable when you need to explain your model's reasoning.

The computational simplicity and interpretability of bag-of-words make it an excellent baseline approach that often performs surprisingly well, even compared to more sophisticated methods. Sometimes the simplest representation that captures the essential information is exactly what you need.